In [ ]:
"""Pruebas unitarias y de integración pequeña sin conexión."""
import tempfile
import unittest
from pathlib import Path

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import indices_diputados_chile as m

In [ ]:
class PipelineTests(unittest.TestCase):
    def temp(self):
        t = tempfile.TemporaryDirectory()
        self.addCleanup(t.cleanup)
        return Path(t.name)

    def fixture(self, directory):
        data = [("1", "Ana Soto Luna", "Periodo_3", "-2", "-2.5", "-1.5", "RN", "", "1"),
                ("2", "Ana Díaz Luna", "2010-14", "0", "-1", "1", "RN", "UDI", "2"),
                ("3", "Luis Reyes Ruiz", "2010", "2", "3", "1", "PC", "", "1"),
                ("2", "Ana Díaz Luna", "Periodo_4", "0", "-1", "1", "UDI", "", "1"),
                ("4", "Pablo Oro Leal", "Periodo_3", "NA", "", "", "", "", "")]
        d = pd.DataFrame(data, columns=[m.ID, "Name", m.P, m.X, "dim_IC_low", "dim_IC_high",
                                       "party_1", "party_2", "quant_parties"])
        d[["party_3", "party_4"]] = ""
        path = directory/"base.csv"
        d.to_csv(path, index=False)
        return path

    def test_periods_and_districts(self):
        for text, canonical in [("PERÍODO 6", "Periodo_6"), ("2022–2026", "Periodo_6"),
                                ("period_4", "Periodo_4"), ("2026", "Periodo_7")]:
            self.assertEqual(m.period(text), canonical)
        self.assertEqual(m.ELECTION[2013], "Periodo_4")
        self.assertEqual(m.ELECTION[2017], "Periodo_5")
        self.assertEqual(m.ELECTION[2025], "Periodo_7")
        with self.assertRaises(ValueError):
            m.period("2025")
        self.assertEqual(m.district_region(19, m.MAIN[0]), 13)
        self.assertEqual(m.district_region(19, m.MAIN[2]), 8)
        self.assertEqual([sum(m.seats(p).values()) for p in m.MAIN], [120, 120, 155, 155])

    def test_mad_dalton_and_accounting(self):
        d = pd.DataFrame({m.X: [-2, 0, 2], "salida": ["A", "A", "B"], m.ID: ["1", "2", "3"]})
        info, parts = m.stats(d, "salida", 2, with_parts=True)
        self.assertAlmostEqual(info["dalton"], np.sqrt(2)/2)
        self.assertAlmostEqual(info["mad_party"], 1.5)
        self.assertAlmostEqual(info["mad_ponderado"], 4/3)
        self.assertAlmostEqual(info["mad_party_normalized"], .75)
        self.assertAlmostEqual(info["var_total"], info["var_entre"]+info["var_intra"])
        self.assertEqual(parts.n.sum(), 3)
        self.assertEqual(info["n_partidos_n1"], 1)
        self.assertTrue(np.isnan(m.jknife(d, "salida", 2)[0]))
        self.assertAlmostEqual(m.jknife(d, "salida", 2)[1], 2/3)

    def test_empty_and_one_party(self):
        d = pd.DataFrame({m.X: [0.0, 0.5], "salida": ["A", "A"]})
        d0 = m.stats(d.iloc[:0], "salida", 1)[0]
        d1 = m.stats(d, "salida", 1)[0]
        self.assertEqual(d0["estado_indice"], "sin_datos")
        self.assertEqual(d1["estado_indice"], "un_partido")
        self.assertTrue(np.isnan(d1["dalton"]))

    def test_panel_and_duplicate(self):
        temp = self.temp()
        file = self.fixture(temp)
        d = m.panel(file)
        self.assertEqual(len(d), 5)
        self.assertEqual(d.loc[1, "salida"], "UDI")
        self.assertTrue(np.isnan(d.loc[1, "unico"]))
        self.assertEqual(d.loc[2, "dim_IC_low_ordenado"], 1)
        self.assertTrue(d.loc[2, "dim_IC_valido"])
        dup = pd.read_csv(file)
        pd.concat([dup, dup.iloc[:1]]).to_csv(file, index=False)
        with self.assertRaisesRegex(ValueError, "duplicada"):
            m.panel(file)

    def test_attendance_and_mixed_optional_sources(self):
        tmp = self.temp()
        long = self.fixture(tmp)
        votes = tmp/"v.csv"
        pd.DataFrame(
            [[1, 1, "Ana", "Soto", "Luna", "Afirmativo", "En Contra", "Abstención", "NA"],
             [2, 2, "Ana", "Díaz", "Luna", "Afirmativo", "No Vota", "", "Dispensado"],
             [3, 3, "Luis", "Reyes", "Ruiz", "na", "Abstención", "Pareado", "Afirmativo"]],
            columns=m.META_VOTOS+["10", "11", "18", "20"]).to_csv(votes, index=False)
        party = tmp/"p.csv"
        pd.DataFrame([["1", "Ana Soto Luna", "RN", "2009-01-01", "2015-01-01", "UDI", "2016-01-01", ""],
                      ["2", "Ana Díaz Luna", "RN", "2009-01-01", "2011-01-01", "UDI", "2011-01-01", ""]],
                     columns=["Id", "nombre_completo", "Alias", "FechaInicio", "FechaTermino",
                              "Alias.1", "FechaInicio.1", "FechaTermino.1"]).to_csv(party, index=False)
        g = tmp/"g.csv"
        pd.DataFrame([[1, "2010", 19], [2, "2010", 19], [3, "2010", 19]],
                     columns=[m.ID, m.P, "Distrito"]).to_csv(g, index=False)
        workbook = tmp/"2013.xlsx"
        pd.DataFrame([["Ana", "Díaz", "Luna", "DIPUTADO", 12, "A"],
                      ["Ana", "Díaz", "Luna", "DIPUTADO", 12, "B"],
                      ["Otro", "Nombre", "Largo", "DIPUTADO", 10, "A"],
                      ["Otro", "Nombre", "Largo", "DIPUTADO", 11, "B"]],
                     columns=["Nombres", "Primer apellido", "Segundo apellido", "Cargo", "Distrito",
                              "Comuna"]).to_excel(workbook, sheet_name="Votación por comuna", startrow=6, index=False)
        a = m.rollcalls(votes, m.MAIN[0])
        self.assertEqual(a.loc[0, "Total_Votaciones"], 4)
        self.assertEqual(a.loc[0, "Asistencia"], .5)
        self.assertEqual(a.loc[0, "Participacion_con_abstencion"], .75)
        self.assertEqual(a.loc[2, "Pareado"], 1)
        b = m.party_dates(party, m.MAIN[0])
        self.assertEqual(b["salida_fechada"].tolist(), ["RN", "UDI"])
        self.assertEqual(b["militancia_solapada"].tolist(), [False, True])
        out = tmp/"out"
        m.main(["--ideologia", str(long), "--matriz", f"2010={votes}", "--militancia", f"2010={party}",
                "--geografia", str(g), "--eleccion", f"2013={workbook}", "--eleccion", f"2025={workbook}",
                "--out", str(out), "--B", "0"])
        a = pd.read_csv(out/"base_composicion.csv")
        self.assertEqual(len(a), 5)
        self.assertTrue(np.isnan(a.loc[3, "Asistencia"]))
        self.assertEqual(a.loc[1, "Region_armonizada"], "Metropolitana")
        ec = pd.read_csv(out/"auditoria_electoral_2013.csv")
        self.assertEqual(ec.Estado_cruce.tolist(), ["candidato_vinculado", "candidatura_ambigua", "candidatura_ambigua"])
        self.assertEqual(pd.read_csv(out/"auditoria_electoral_2025.csv").iloc[0]["Estado_cruce"], "sin_ideologia_publicada")
        self.assertEqual(pd.read_csv(out/"panel_regional_comun.csv").atribucion.unique().tolist(), ["salida"])

    def test_geography_and_name_validation(self):
        temp = self.temp()
        g = temp/"g.csv"
        pd.DataFrame([[1, "2018", "19.0"], [2, "2018", ""]],
                     columns=[m.ID, m.P, "Distrito"]).to_csv(g, index=False)
        a = m.geography(g)
        self.assertEqual(a.loc[0, "Region"], "Ñuble")
        self.assertEqual(a.loc[0, "Region_armonizada"], "Biobío y Ñuble")
        self.assertTrue(pd.isna(a.loc[1, "Distrito"]))
        initial = m.panel(self.fixture(temp))
        addon = initial.loc[[0], m.KEY].assign(Name_matriz="Otra Persona")
        with self.assertRaisesRegex(ValueError, "nombres distintos"):
            m.validate_name_ids(initial, addon, "Name_matriz", "auditoría")

    def test_bootstrap_joint_id_reproducible(self):
        a = pd.DataFrame({m.ID: [str(i) for i in range(20)],
                          m.X: np.linspace(-2, 2, 20), "partido": ["A"]*10+["B"]*10})
        b = a.assign(dim_ideology=a[m.X]*.75)
        def run():
            return m.contrast([a, b], "partido", 2, [-1, 1], np.random.default_rng(9), 199)
        self.assertEqual(run(), run())
        self.assertEqual(run()["B_validas"], 199)
        self.assertEqual(run()["n_id_union"], 20)
        self.assertLess(run()["delta"], 0)

In [ ]:
if __name__ == "__main__":
    unittest.main()